*<small>Last updated: 2026-10-01 18:06:24 UTC | Student Version (No Solutions)</small>*

# Clustering: K-Means, Hierarchical, and Gaussian Mixture Models [SOLUTIONS]

**Course**: Machine Learning Labs
**Lab**: 08 — Clustering

---

## Table of Contents

0. [Setup and Configuration](#0)
1. [Introduction to Clustering](#1)
2. [Dataset 1: Online Retail & RFM Feature Engineering](#2)
3. [K-Means Clustering](#3)
4. [Hierarchical Clustering](#4)
5. [Gaussian Mixture Models](#5)
6. [Comparative Analysis on Online Retail](#6)
7. [Dataset 2: HAR & External Evaluation (ARI / NMI)](#7)
8. [Real-World Considerations](#8)
9. [Final Challenges](#9)

---

**Estimated Time**: 3–4 hours
**Prerequisites**: Unsupervised learning basics, scikit-learn, NumPy, Pandas

**Note**: This is the SOLUTIONS version containing all exercise answers.

## 🚚 Deliverables

**Don't forget to duplicate this notebook to be able to edit: File -> Save a copy in Drive**

In this lab, you do not need to write a report in a separate document. Please complete all required
activities within this Google Colab notebook. Remember that a notebook allows you to enter text
elements similarly to a document processor. Once the proposed activities are completed, you must
submit the following on the platform:

1. A PDF file generated in Google Colab from the menu "File" -> "Print".

2. The public Google Colab link. To do so, go to the share button and change the sharing settings
   to "Anyone with the link".

## 📔 Resolution

After the deadline, you will find the resolution at the following Notion link:

https://app.notion.com/p/R-Clustering-46264f3ff1004fc8ad39badb0ed7c86f?source=copy_link

Remember that there are several ways to solve the proposed activities in the labs.
Use the provided resolution as a reference. If the resolution link is not public,
request access at the same Notion link.

## Section 0: Setup and Configuration <a id='0'></a>

In [ ]:
try:
    import google.colab
    IN_COLAB = True
    import subprocess
    subprocess.run(["pip", "install", "-q", "openpyxl"], check=True)
except ImportError:
    IN_COLAB = False

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.datasets import fetch_openml, load_sample_image
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score,
)
from sklearn.utils import shuffle

import sklearn
warnings.filterwarnings('ignore')
print(f"NumPy {np.__version__} | Pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

In [ ]:
# ── Global configuration ─────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ONLINE_RETAIL_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases"
    "/00352/Online%20Retail.xlsx"
)

plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', palette='tab10')

## Section 1: Introduction to Clustering <a id='1'></a>

**Clustering** is an unsupervised learning task: given unlabeled data, discover
groups of similar observations — without any prior knowledge of what those groups
should be.

| Property | Hard assignment | Soft assignment |
|---|---|---|
| Each point belongs to | exactly one cluster | all clusters (with probabilities) |
| Algorithms | K-Means, Hierarchical | Gaussian Mixture Models |
| Output | integer label | probability vector |

### Real-world applications

- **Customer segmentation**: group buyers by purchasing behaviour (Online Retail)
- **Activity detection**: identify motion patterns from smartphone sensors (HAR)
- **Document grouping**: cluster news articles by topic

### This lab

We use **two datasets** with distinct goals:

1. **Online Retail (UCI)** — no ground-truth labels.
   We engineer RFM features and compare K-Means, Hierarchical, and GMM using
   *internal* metrics (silhouette, Davies-Bouldin, Calinski-Harabasz).

2. **Human Activity Recognition (HAR)** — 6 known activity labels.
   We measure clustering quality using *external* metrics (ARI, NMI) and
   compare them to what internal metrics alone would suggest.

## Section 2: Dataset 1 — Online Retail & RFM Feature Engineering <a id='2'></a>

The [Online Retail dataset (UCI)](https://archive.ics.uci.edu/ml/datasets/Online+Retail)
contains ~541 k invoice records from a UK-based online retailer (Dec 2010 – Dec 2011).
Each row is one product line within a transaction (invoice number, stock code,
quantity, unit price, timestamp, customer ID, country).

**The problem**: this is a *transaction-level* log, but we want to cluster
*customers*. Clustering raw transactions would group purchases, not people — so
before any model touches the data, we must aggregate the log into one row per
customer. The standard, battle-tested way to summarise customer behaviour for
segmentation is the **RFM** framework.

### What is RFM?

RFM distils a customer's entire purchase history into three interpretable
behavioural signals. Each is computed by grouping all of a customer's transactions
and applying a single aggregation:

| Feature | Question it answers | How we compute it | Business meaning |
|---|---|---|---|
| **R**ecency | *How long since the customer last bought?* | days between the customer's most recent invoice and a fixed reference date | low Recency → recently active; high Recency → at risk of churn |
| **F**requency | *How often does the customer buy?* | count of **distinct invoices** per customer | high Frequency → engaged, loyal, habitual buyer |
| **M**onetary | *How much does the customer spend?* | sum of `Quantity × UnitPrice` across all their lines | high Monetary → high-value customer worth retaining |

The classic marketing intuition: the *best* customers bought **recently**, buy
**often**, and **spend** a lot. RFM turns that intuition into three numbers that
a clustering algorithm can act on.

### Modelling decisions baked into the features

A few choices matter and are easy to get wrong:

- **Reference date for Recency.** "Today" is meaningless for a historical dataset,
  so we anchor Recency to the day *after* the last transaction in the data
  (`max(InvoiceDate) + 1 day`). This guarantees every customer has a Recency of at
  least 1 day and makes the feature reproducible.
- **Frequency = distinct invoices, not line items.** A single basket of 20 products
  is *one* purchase event, not 20. Counting unique `InvoiceNo` values captures
  *shopping trips*; counting rows would just reward large baskets.
- **Monetary uses revenue per line.** We pre-computed `TotalPrice = Quantity ×
  UnitPrice` during cleaning, then sum it per customer to get lifetime spend.

### Why preprocessing is not optional here

RFM features are wildly different in scale and shape, which breaks distance-based
clustering if left untreated:

- **Different units and ranges.** Recency is in *days* (1–374), Frequency is a
  *count* (1–209), Monetary is in *pounds* (£3 to £280 k). Without
  standardisation, Monetary's magnitude would dominate every Euclidean distance and
  K-Means would essentially cluster on spend alone.
- **Heavy right skew.** A small number of "whale" customers buy enormously more
  than the median, producing long right tails (especially in Frequency and
  Monetary). Skew pulls cluster centroids toward outliers. We apply a
  **`log1p` transform** to Frequency and Monetary to compress those tails toward
  symmetry, then verify the effect numerically with a skewness table (Recency is
  already roughly symmetric, so we leave it untransformed).
- **Order of operations.** Aggregate → log-transform the skewed features →
  **standardise** (zero mean, unit variance). Scaling must come *last* so every
  feature contributes comparably to the distance metric.

**Goal of this section**: turn the raw 541 k-row transaction log into a clean,
4 338-customer RFM table that is correctly transformed and scaled — the input on
which we will compare K-Means, Hierarchical, and GMM in the sections that follow.

In [ ]:
# Load — try local cache first, fall back to URL
local_path = os.path.join("assets", "Online_Retail.xlsx")
if os.path.exists(local_path):
    df_raw = pd.read_excel(local_path, engine="openpyxl")
else:
    print("Downloading Online Retail dataset (~23 MB)…")
    df_raw = pd.read_excel(ONLINE_RETAIL_URL, engine="openpyxl")
    os.makedirs("assets", exist_ok=True)
    df_raw.to_excel(local_path, index=False)
    print("Saved to assets/Online_Retail.xlsx for future runs.")

print(f"Raw shape: {df_raw.shape}")
df_raw.head()

In [ ]:
# Basic EDA
print(f"Date range: {df_raw['InvoiceDate'].min()} → {df_raw['InvoiceDate'].max()}")
print(f"Unique customers: {df_raw['CustomerID'].nunique():,}")
print(f"Unique products:  {df_raw['StockCode'].nunique():,}")
print(f"\nMissing values:\n{df_raw.isnull().sum()}")

In [ ]:
# Clean: remove returns, missing customers, zero/negative prices
df = df_raw.dropna(subset=['CustomerID']).copy()
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['CustomerID'] = df['CustomerID'].astype(int)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print(f"After cleaning: {len(df):,} rows | {df['CustomerID'].nunique():,} customers")

In [ ]:
# RFM feature engineering
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum'),
).reset_index()

print(f"RFM table shape: {rfm.shape}")
rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2)

In [ ]:
# Log-transform Frequency and Monetary to reduce right skew.
# Recency is already roughly symmetric, so we leave it untransformed.
log_features = ['Frequency', 'Monetary']

rfm_log = rfm[['Recency', 'Frequency', 'Monetary']].copy()
rfm_log[log_features] = np.log1p(rfm_log[log_features])

# Quantify skewness to justify the transform (rule of thumb: |skew| > 1 = strong,
# 0.5–1 = moderate, < 0.5 = approximately symmetric)
skewness = pd.DataFrame({
    'Skew (raw)':         rfm[['Recency', 'Frequency', 'Monetary']].skew(),
    'Skew (transformed)': rfm_log[['Recency', 'Frequency', 'Monetary']].skew(),
}).round(2)
print("Skewness before vs after log transform:")
print(skewness.to_string())

# Standardise — mandatory before any distance-based algorithm
scaler = StandardScaler()
X = scaler.fit_transform(rfm_log)
print(f"\nScaled feature matrix: {X.shape}  mean≈{X.mean():.2f}  std≈{X.std():.2f}")

### Exercise 2.1 — RFM Distributions (Basic)

Plot the raw and log-transformed distributions for all three RFM features in a
2 × 3 grid (top row = raw, bottom row = transformed).

**What to observe**: Does the log transform bring the distributions closer to
symmetric? Which feature benefits the most?

In [ ]:
# Exercise 2.1 — RFM distributions: raw (top) vs log1p-transformed (bottom)
rfm_feats = ['Recency', 'Frequency', 'Monetary']
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
skew_rows = {}
for j, feat in enumerate(rfm_feats):
    raw = rfm[feat]
    # rfm_log already holds log1p(Frequency) and log1p(Monetary); Recency is kept raw there,
    # so for the comparison we apply log1p to it here as well.
    logged = rfm_log[feat] if feat in log_features else np.log1p(rfm[feat])
    skew_rows[feat] = {'skew raw': raw.skew(), 'skew log1p': logged.skew()}

    axes[0, j].hist(raw, bins=60, color='steelblue', edgecolor='white')
    axes[0, j].set_title(f'{feat} (raw) — skew = {raw.skew():.2f}')
    axes[1, j].hist(logged, bins=60, color='coral', edgecolor='white')
    axes[1, j].set_title(f'log1p({feat}) — skew = {logged.skew():.2f}')
    axes[1, j].set_xlabel(f'log1p({feat})')
    axes[0, j].set_xlabel(feat)
axes[0, 0].set_ylabel('Customers'); axes[1, 0].set_ylabel('Customers')
plt.suptitle('RFM distributions — top: raw, bottom: log1p-transformed', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

skew_df = pd.DataFrame(skew_rows).T
skew_df['|skew| reduction'] = skew_df['skew raw'].abs() - skew_df['skew log1p'].abs()
print(skew_df.round(2).to_string())
print(f"\nFeature that gains most symmetry with log1p: {skew_df['|skew| reduction'].idxmax()}")

**Observation.** Frequency and Monetary are extremely right-skewed in raw form: most customers place a few orders and spend a few hundred pounds, while a handful of wholesale-like customers place hundreds of orders and spend tens/hundreds of thousands, which compresses the histogram into a single bar near zero. `log1p` turns both into roughly bell-shaped distributions. **Monetary benefits the most** (its |skew| drops the most — see the printed table), followed by Frequency, which remains somewhat skewed because it is a discrete count with a large mass at 1 order. Recency is only moderately skewed; applying `log1p` to it over-corrects and creates a left skew, which is why the notebook keeps Recency untransformed.

## Section 3: K-Means Clustering <a id='3'></a>

### Algorithm recap

**Lloyd's algorithm**:
1. Randomly initialise k centroids (or use K-Means++ for smarter placement)
2. Assign each point to the nearest centroid (hard assignment)
3. Recompute centroids as cluster means
4. Repeat 2–3 until centroids stop moving

**Inertia** (within-cluster sum of squared distances) decreases monotonically
with k — so we use the *elbow* heuristic to pick a sensible k.
**Silhouette score** provides a second opinion: higher is better, range [−1, 1].

### Choosing k: elbow + silhouette

In [ ]:
k_range = range(2, 11)
inertias, sil_scores = [], []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, 'o-', color='steelblue', linewidth=2)
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia (within-cluster SSE)')
axes[0].set_title('Elbow Method')
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_range, sil_scores, 'o-', color='coral', linewidth=2)
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Choosing k for K-Means on RFM data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Fit final model with chosen k
K_BEST = 3  # adjust if your elbow/silhouette suggests a different value

km_final = KMeans(n_clusters=K_BEST, init='k-means++', n_init=10, random_state=RANDOM_STATE)
rfm['Cluster_KM'] = km_final.fit_predict(X)

print(f"Cluster sizes (K-Means, k={K_BEST}):")
print(rfm['Cluster_KM'].value_counts().sort_index().to_string())

In [ ]:
# PCA projection for 2-D visualisation
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca2.fit_transform(X)
explained = pca2.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                c=rfm['Cluster_KM'], cmap='tab10', alpha=0.6, s=25)
ax.set_xlabel(f'PC1 ({explained[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({explained[1]:.1%} variance)')
ax.set_title(f'K-Means clusters (k={K_BEST}) in PCA space')
plt.colorbar(sc, ax=ax, label='Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# Cluster profiles — business interpretation
profile = rfm.groupby('Cluster_KM').agg(
    Count=('CustomerID', 'count'),
    Recency_days=('Recency', 'mean'),
    Frequency_orders=('Frequency', 'mean'),
    Monetary_gbp=('Monetary', 'mean'),
).round(1)
profile['% customers'] = (profile['Count'] / len(rfm) * 100).round(1)
profile

### Exercise 3.1 — Effect of k (Basic)

Fit K-Means for k ∈ {2, 3, 4, 5, 6, 7} and display each result as a PCA
scatter plot in a 2 × 3 grid. Include the silhouette score in each subplot title.

**What to observe**: At what point do clusters become too fragmented to be
business-meaningful?

In [ ]:
# Exercise 3.1 — Effect of k (PCA projection, silhouette in the title)
k_values = [2, 3, 4, 5, 6, 7]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
size_rows = {}
for ax, k in zip(axes.flat, k_values):
    km_k = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    labels_k = km_k.fit_predict(X)
    sil_k = silhouette_score(X, labels_k)
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_k, cmap='tab10', s=8, alpha=0.6)
    ax.set_title(f'k = {k}  |  silhouette = {sil_k:.3f}')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    shares = np.bincount(labels_k) / len(labels_k) * 100
    size_rows[k] = {'silhouette': sil_k, 'smallest cluster (%)': shares.min(), 'largest cluster (%)': shares.max()}
plt.suptitle('K-Means on RFM for different k (PCA 2-D projection)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(pd.DataFrame(size_rows).T.round(3).to_string())

**Observation.** For k = 2 and k = 3 the clusters are large, contiguous bands along PC1 (the "value" axis: recent, frequent, high-spending customers on one side and lapsed low-value customers on the other) and the silhouette is highest. From **k = 4–5 onwards** the new clusters are obtained by slicing the same continuous cloud into thinner strips, the silhouette keeps falling and small segments appear (see the "smallest cluster %" column). These extra segments differ only by small shifts in recency/spend rather than by a different behaviour, so they are hard to name and to target with a different campaign. For a marketing team, k = 3 (at most 4) is the last value that yields clearly distinct, actionable segments; k ≥ 5 is too fragmented to be business-meaningful.

### Exercise 3.2 — Initialisation Strategies (Intermediate)

Run K-Means 10 times using `n_init=1` with both `init='random'` and
`init='k-means++'`. For each trial record the final inertia and number of
iterations. Plot side-by-side boxplots.

**Key question**: Which initialisation is more stable? Which converges faster?

In [ ]:
# Exercise 3.2 — Initialisation strategies: 10 runs with n_init=1
init_rows = []
for init in ['random', 'k-means++']:
    for trial in range(10):
        km_i = KMeans(n_clusters=K_BEST, init=init, n_init=1, random_state=RANDOM_STATE + trial).fit(X)
        init_rows.append({'init': init, 'trial': trial, 'inertia': km_i.inertia_, 'n_iter': km_i.n_iter_})
df_init = pd.DataFrame(init_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=df_init, x='init', y='inertia', ax=axes[0], palette='Set2')
sns.stripplot(data=df_init, x='init', y='inertia', ax=axes[0], color='k', size=4)
axes[0].set_title('Final inertia (10 runs, n_init=1)')
sns.boxplot(data=df_init, x='init', y='n_iter', ax=axes[1], palette='Set2')
sns.stripplot(data=df_init, x='init', y='n_iter', ax=axes[1], color='k', size=4)
axes[1].set_title('Iterations to converge (10 runs, n_init=1)')
plt.tight_layout()
plt.show()

print(df_init.groupby('init')[['inertia', 'n_iter']].agg(['mean', 'std', 'min', 'max']).round(2).to_string())

**Conclusion.** `k-means++` is both **more stable** and **faster**: its 10 runs end at practically the same (lowest) inertia with a very small spread, and it needs fewer Lloyd iterations on average, because its seeding already places the initial centroids far apart and close to the final solution. `init='random'` picks initial centroids uniformly among the points; some runs start with two centroids in the same dense region and either converge to a worse local optimum (higher inertia outliers) or need more iterations to move the centroids apart. This is why scikit-learn uses `k-means++` by default and why, with random initialisation, a larger `n_init` is necessary.

## Section 4: Hierarchical Clustering <a id='4'></a>

**Agglomerative** hierarchical clustering starts with each point as its own cluster
and merges the two closest clusters at each step. The merge history is visualised
as a **dendrogram**.

The *linkage criterion* defines "distance between clusters":

| Linkage | Definition | Tendency |
|---|---|---|
| Single | min pairwise distance | chaining, elongated clusters |
| Complete | max pairwise distance | compact, similar-size clusters |
| Average | mean pairwise distance | compromise |
| **Ward** | minimise within-cluster variance | spherical, balanced (default choice) |

**Complexity**: O(n² log n) — we subsample to 400 customers for the dendrogram.

In [ ]:
# Dendrogram on a subsample for readability
np.random.seed(RANDOM_STATE)
sub_idx = np.random.choice(len(X), size=min(400, len(X)), replace=False)
X_sub = X[sub_idx]

Z = linkage(X_sub, method='ward')

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(Z, ax=ax, truncate_mode='lastp', p=20,
           leaf_rotation=45, leaf_font_size=10, show_contracted=True)
ax.set_title(f'Dendrogram — Ward linkage (400-customer subsample)')
ax.set_xlabel('Cluster (sample count in parentheses)')
ax.set_ylabel('Ward distance')
plt.tight_layout()
plt.show()

In [ ]:
# Fit on full dataset and compare to K-Means
agg = AgglomerativeClustering(n_clusters=K_BEST, linkage='ward')
rfm['Cluster_Agg'] = agg.fit_predict(X)

sil_km  = silhouette_score(X, rfm['Cluster_KM'])
sil_agg = silhouette_score(X, rfm['Cluster_Agg'])
print(f"Silhouette — K-Means: {sil_km:.4f}  |  Hierarchical (Ward): {sil_agg:.4f}")
print(f"\nCluster sizes (Hierarchical):")
print(rfm['Cluster_Agg'].value_counts().sort_index().to_string())

### Exercise 4.1 — Linkage Comparison (Intermediate)

Plot dendrograms for Ward, Complete, Average, and Single linkage in a 2 × 2
grid (same 400-customer subsample). Below each dendrogram, cut the tree at
`k = K_BEST` and print the resulting cluster sizes.

**Key question**: Which linkage produces the most balanced clusters?
Why does Single linkage behave differently?

In [ ]:
# Exercise 4.1 — Linkage comparison on the same 400-customer subsample
linkages = ['ward', 'complete', 'average', 'single']
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
link_sizes = {}
for ax, method in zip(axes.flat, linkages):
    Z_m = linkage(X_sub, method=method)
    dendrogram(Z_m, ax=ax, truncate_mode='lastp', p=20, leaf_rotation=45, leaf_font_size=9, show_contracted=True)
    labels_m = fcluster(Z_m, K_BEST, criterion='maxclust')
    sizes = np.bincount(labels_m)[1:]
    link_sizes[method] = sorted(sizes.tolist(), reverse=True)
    ax.set_title(f'{method.capitalize()} linkage')
    ax.set_ylabel('Distance')
    ax.set_xlabel(f'Cut at k={K_BEST} -> cluster sizes: {sorted(sizes.tolist(), reverse=True)}', fontsize=11)
    print(f"{method:>8s}: cluster sizes at k={K_BEST} -> {sorted(sizes.tolist(), reverse=True)}")
plt.suptitle('Dendrograms by linkage criterion (400-customer subsample)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Observation.** **Ward** produces the most balanced clusters, since each merge minimises the increase in within-cluster variance, which naturally yields compact groups of comparable size; **complete** linkage is the next most balanced. **Average** and especially **single** linkage give very unbalanced cuts: one giant cluster containing almost all customers plus one or two tiny clusters made of a few outliers (e.g. the extreme high-spenders). Single linkage behaves differently because of **chaining**: the distance between two clusters is the distance between their *closest* pair of points, so in a continuous cloud like RFM (no empty gaps between groups) clusters keep growing by absorbing the nearest neighbouring point one at a time. The dendrogram shows many merges at very small, similar heights, and the only large jumps are those of isolated outliers — so cutting at k = 3 just separates outliers from everyone else.

## Section 5: Gaussian Mixture Models <a id='5'></a>

A **Gaussian Mixture Model** assumes the data is generated by a mixture of k
Gaussian distributions. The **EM algorithm** iterates between:

- **E-step**: compute the *posterior probability* that each point belongs to each
  component (soft assignment)
- **M-step**: update means, covariances, and mixing weights to maximise likelihood

The *covariance type* controls model flexibility:

| Type | Parameters | Flexibility |
|---|---|---|
| `spherical` | one variance per component | lowest |
| `diag` | diagonal covariance per component | medium |
| `tied` | one full covariance shared by all | medium |
| `full` | full covariance per component | highest |

**Model selection**: BIC and AIC penalise model complexity (lower is better).

In [ ]:
gmm = GaussianMixture(n_components=K_BEST, covariance_type='full',
                      random_state=RANDOM_STATE, n_init=5)
gmm.fit(X)
rfm['Cluster_GMM'] = gmm.predict(X)
proba = gmm.predict_proba(X)
rfm['Max_Posterior'] = proba.max(axis=1)

print(f"Converged: {gmm.converged_}  |  BIC: {gmm.bic(X):.1f}  |  AIC: {gmm.aic(X):.1f}")
print(f"\nCluster sizes (GMM):")
print(rfm['Cluster_GMM'].value_counts().sort_index().to_string())

In [ ]:
# Visualise assignments vs uncertainty
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sc1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                       c=rfm['Cluster_GMM'], cmap='tab10', alpha=0.6, s=20)
axes[0].set_title('GMM — Hard assignments (argmax posterior)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
plt.colorbar(sc1, ax=axes[0], label='Cluster')

sc2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                       c=rfm['Max_Posterior'], cmap='RdYlGn',
                       alpha=0.7, s=20, vmin=0.5, vmax=1.0)
axes[1].set_title('GMM — Assignment certainty (max posterior)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.colorbar(sc2, ax=axes[1], label='Max Posterior')

plt.tight_layout()
plt.show()

In [ ]:
# BIC / AIC curves for model selection
k_range_gmm = range(2, 11)
bic_scores, aic_scores = [], []

for k in k_range_gmm:
    g = GaussianMixture(n_components=k, covariance_type='full',
                        random_state=RANDOM_STATE, n_init=3)
    g.fit(X)
    bic_scores.append(g.bic(X))
    aic_scores.append(g.aic(X))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_range_gmm, bic_scores, 'o-', label='BIC', color='steelblue', linewidth=2)
ax.plot(k_range_gmm, aic_scores, 's--', label='AIC', color='coral', linewidth=2)
ax.axvline(x=K_BEST, color='gray', linestyle=':', alpha=0.7, label=f'k={K_BEST} (chosen)')
ax.set_xlabel('Number of components k')
ax.set_ylabel('Score (lower = better)')
ax.set_title('BIC and AIC for GMM Model Selection')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reading the BIC / AIC curves

Both BIC and AIC reward goodness-of-fit (higher likelihood) but **penalise model
complexity** — every extra component adds parameters (a mean, a covariance, a
mixing weight), so the penalty grows with k. BIC penalises more harshly than AIC,
which is why the two curves separate slightly at large k. For both, **lower is
better**, and the textbook recipe is to pick the k at the *minimum*, or where the
curve stops dropping meaningfully.

**What this figure actually shows is a cautionary case:**

- BIC and AIC sit almost on top of each other and **keep decreasing all the way to
  k = 10** (down to roughly −8 000 / −9 000). There is **no true minimum** in the
  range — read literally, "lowest score wins" would select k = 10.
- The decline is **steepest from k = 2 → 4**, then **flattens dramatically after
  k ≈ 5**. Each component beyond that buys very little extra likelihood.

This is a classic GMM pitfall: with flexible (full) covariance, the model can
always raise its likelihood a little by adding a component to chase fine structure
— often **overfitting** noise rather than discovering real segments. The complexity
penalty here is not strong enough to overcome that, so the curve never bottoms out.

### Why k = 3 is still a good choice

- **It sits at the elbow.** The dramatic drop happens by k ≈ 4 and the curve is
  essentially flat afterward. k = 3 captures the bulk of the achievable likelihood
  gain before diminishing returns set in — the same elbow logic used for K-Means
  inertia. Adding components past this point yields tiny, hard-to-interpret splits.
- **It agrees with every other diagnostic.** The K-Means silhouette peaks at k = 3,
  the Ward dendrogram's largest gaps support 2–3 clusters, and the full grid search
  in Section 6 ranks k = 3 first for all methods. BIC/AIC alone are ambiguous here,
  but **consensus across independent methods** is strong evidence.
- **It is interpretable and actionable.** k = 3 maps cleanly onto the
  Champions / Regular / Lapsed RFM segments — a story a marketing team can act on.
  The k = 10 solution that minimises BIC would produce tiny fragments with no
  business meaning.

**The lesson:** BIC and AIC are *guides, not oracles*. When the curve declines
monotonically, do not blindly take the global minimum — combine it with the elbow
location, agreement across methods, and interpretability. Here all three point to
**k = 3**, even though the raw BIC/AIC numbers do not single it out.

### Exercise 5.1 — Covariance Type Comparison (Intermediate)

Plot BIC vs k for all four covariance types (`full`, `tied`, `diag`,
`spherical`) on the same axes. Then print, for each type, the k that minimises
BIC.

**Key question**: Does `full` always win? What is the trade-off with
`spherical` or `diag`?

In [ ]:
# Exercise 5.1 — BIC vs k for every covariance type
cov_types = ['full', 'tied', 'diag', 'spherical']
k_range_cov = range(2, 11)
bic_by_type = {}
for ct in cov_types:
    bic_by_type[ct] = [GaussianMixture(n_components=k, covariance_type=ct, random_state=RANDOM_STATE, n_init=3)
                       .fit(X).bic(X) for k in k_range_cov]

fig, ax = plt.subplots(figsize=(10, 6))
for ct, marker in zip(cov_types, ['o', 's', '^', 'D']):
    ax.plot(list(k_range_cov), bic_by_type[ct], marker=marker, linewidth=2, label=ct)
ax.set_xlabel('Number of components k'); ax.set_ylabel('BIC (lower = better)')
ax.set_title('GMM model selection — BIC by covariance type')
ax.legend(title='covariance_type'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

bic_df = pd.DataFrame(bic_by_type, index=list(k_range_cov))
for ct in cov_types:
    print(f"{ct:>9s}: k minimising BIC = {bic_df[ct].idxmin():2d}  (BIC = {bic_df[ct].min():.1f})")
print(f"\nOverall best: covariance_type = '{bic_df.min().idxmin()}', k = {bic_df[bic_df.min().idxmin()].idxmin()}")

**Does `full` always win?** On this data `full` reaches the lowest BIC values: the RFM features are correlated (frequent customers also spend more), so components with their own rotated, elongated covariance describe the data with far fewer components than axis-aligned (`diag`) or round (`spherical`) Gaussians, and `tied` is limited by sharing a single shape across all components. However, `full` does not win at every k by construction: BIC penalises its extra parameters (6 covariance parameters per component in 3-D vs 3 for `diag` and 1 for `spherical`), and the minimum of the restrictive types is usually reached at a larger k, because they need more components to approximate the same tilted density. **Trade-off:** `spherical`/`diag` are cheaper, more stable with few samples or many dimensions and less prone to degenerate covariances, but they impose a shape that does not match correlated features; `full` is the most flexible and fits better, at the cost of more parameters (O(d²) per component), more risk of overfitting and slower fitting in high dimensions.

### Exercise 5.2 — Uncertain Customers (Advanced)

Identify customers whose max posterior probability is below 0.60 — these are
customers that GMM could not confidently assign to any cluster.

1. Print the count and percentage of uncertain customers.
2. Compare their mean RFM values to the rest.
3. Plot uncertain customers in red over the PCA scatter of all customers.

**What do uncertain customers have in common?**

In [ ]:
# Exercise 5.2 — Uncertain customers (max posterior < 0.60)
THRESH = 0.60
uncertain = rfm['Max_Posterior'] < THRESH
n_unc = int(uncertain.sum())
print(f"Uncertain customers: {n_unc} of {len(rfm)} ({n_unc / len(rfm):.1%})")

comp = rfm.groupby(uncertain.map({True: 'Uncertain (<0.60)', False: 'Confident (>=0.60)'}))[
    ['Recency', 'Frequency', 'Monetary']].agg(['mean', 'median']).round(1)
print("\nRFM comparison (mean and median):")
print(comp.to_string())
print("\nGMM cluster of the uncertain customers:")
print(rfm.loc[uncertain, 'Cluster_GMM'].value_counts().sort_index().to_string())
# Which pair of components are they torn between?
top2 = np.argsort(-proba, axis=1)[:, :2]
pairs = pd.Series([tuple(sorted(p)) for p in top2[uncertain.values]]).value_counts()
print("\nMost common pair of competing components:")
print(pairs.head().to_string())

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_pca[~uncertain.values, 0], X_pca[~uncertain.values, 1], c='lightgrey', s=12, alpha=0.6,
           label=f'Confident (max posterior >= {THRESH})')
ax.scatter(X_pca[uncertain.values, 0], X_pca[uncertain.values, 1], c='red', s=18, alpha=0.9,
           label=f'Uncertain (max posterior < {THRESH}), n={n_unc}')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('GMM — uncertain customers highlighted in the PCA projection')
ax.legend(); plt.tight_layout(); plt.show()

**What do uncertain customers have in common?** They are a small fraction of the customer base and, in the PCA plot, they lie on the **boundaries between GMM components** rather than in the cores of the clusters. Their RFM profile is **intermediate**: their mean/median Recency, Frequency and Monetary values sit between those of the clear "champion" and "lapsed" groups (moderately recent, a few orders, mid-level spend), and almost all of them are torn between the same two neighbouring components (see the most common pair). In business terms they are customers in transition — e.g. regular buyers starting to lapse, or new customers who may become loyal — which makes them a natural target for nurturing/retention campaigns. This information is only available because GMM gives soft assignments; K-Means would silently put them in one cluster.

## Section 6: Comparative Analysis on Online Retail <a id='6'></a>

With all three methods fitted, we can now compare them side by side using
**internal** evaluation metrics — metrics that do not require ground-truth labels.

| Metric | Direction | Measures |
|---|---|---|
| Silhouette score | ↑ higher is better | separation vs cohesion |
| Davies-Bouldin index | ↓ lower is better | avg ratio of intra-to-inter cluster distance |
| Calinski-Harabasz index | ↑ higher is better | ratio of between-to-within cluster dispersion |

In [ ]:
methods_rfm = {
    'K-Means':             rfm['Cluster_KM'],
    'Hierarchical (Ward)': rfm['Cluster_Agg'],
    'GMM':                 rfm['Cluster_GMM'],
}

rows = []
for name, labels in methods_rfm.items():
    rows.append({
        'Method': name,
        'Silhouette ↑':        silhouette_score(X, labels),
        'Davies-Bouldin ↓':    davies_bouldin_score(X, labels),
        'Calinski-Harabasz ↑': calinski_harabasz_score(X, labels),
    })

df_comparison = pd.DataFrame(rows).set_index('Method').round(4)
df_comparison

In [ ]:
# Side-by-side PCA scatter plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles    = ['K-Means', 'Hierarchical (Ward)', 'GMM']
col_names = ['Cluster_KM', 'Cluster_Agg', 'Cluster_GMM']

for ax, title, col in zip(axes, titles, col_names):
    sc = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                    c=rfm[col], cmap='tab10', alpha=0.5, s=15)
    ax.set_title(title)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    plt.colorbar(sc, ax=ax, label='Cluster')

plt.suptitle('Cluster assignments across methods (PCA projection)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Why these metrics are unfair to GMM

The table ranks GMM well below K-Means and Hierarchical on all three metrics
(Silhouette 0.19 vs 0.42, Davies-Bouldin 1.57 vs 0.82, Calinski-Harabasz 1891 vs
4395). It is tempting to conclude GMM is the worst method here — but that
conclusion is wrong. The problem is a **mismatch between what these metrics reward
and what GMM optimises**, not a failure of GMM.

**All three internal metrics have a built-in bias toward compact, spherical, hard
clusters:**

- They are computed from **Euclidean distances to centroids**. Silhouette compares
  each point's distance to its own cluster vs the nearest other cluster;
  Davies-Bouldin and Calinski-Harabasz both rely on within- vs between-cluster
  *distance/dispersion* ratios. All implicitly assume clusters are round blobs of
  similar size.
- **K-Means optimises exactly this geometry.** Its objective *is* within-cluster
  squared distance, so it is almost guaranteed to score best on metrics built from
  the same quantity — it is being judged on its home turf. Ward linkage minimises
  the same variance criterion, so it lands a close second.
- **GMM does not optimise geometry at all.** It maximises *likelihood* under a
  probabilistic model, and it is free to fit **elongated, rotated, overlapping
  clusters of unequal size** (full covariance). Points in the overlap region get a
  low silhouette even when the model has assigned them correctly with a sensible
  posterior. The metric penalises overlap; GMM is designed to *model* overlap.

In short: **a low Silhouette/DB/CH for GMM measures how non-spherical and
overlapping its clusters are, not how wrong they are.** Judging GMM with these
metrics is like grading a probabilistic model with a ruler.

### How to evaluate GMM fairly

Match the evaluation to the model's objective:

| Tool | What it measures | Why it suits GMM |
|---|---|---|
| **BIC / AIC** | likelihood penalised for complexity | scores GMM on its own objective (see Section 5) |
| **Log-likelihood** | fit of the generative model | native quantity GMM maximises |
| **Posterior confidence** | per-point assignment certainty | exploits soft assignment (see Exercise 5.2) |
| **External metrics (ARI/NMI)** | agreement with ground truth | bias-free when labels exist (see Section 7) |

### The advantages of GMM these metrics cannot see

- **Soft (probabilistic) assignment.** Every customer gets a probability vector
  over clusters, not a single hard label. This surfaces *borderline* customers
  (Exercise 5.2 found ~6.8% with max posterior < 0.6) — information K-Means
  discards by forcing every point to 100% membership.
- **Flexible cluster shapes.** With full covariance, clusters can be elliptical,
  correlated, and differently sized/oriented — a far better fit when segments are
  not round, which is the norm for real RFM data.
- **A generative, probabilistic model.** Because GMM defines a density, you can
  evaluate the likelihood of new points, sample synthetic customers, detect
  outliers (low likelihood), and select k with principled criteria (BIC/AIC)
  instead of heuristics.
- **Handles overlapping segments.** Real customer segments blend into one another;
  GMM models that blend explicitly rather than drawing artificial hard boundaries
  through dense regions.

**Bottom line:** on this dataset K-Means produces the most geometrically compact
*hard* partition, which is exactly what Silhouette/DB/CH reward — so it "wins" the
table. But if the goal is **uncertainty-aware segmentation, non-spherical clusters,
or a generative model of customer behaviour**, GMM is the stronger tool, and it
should be judged by BIC/AIC and posterior confidence rather than by centroid-based
internal metrics.

### Exercise 6.1 — Full Grid Search (Advanced)

For each k ∈ {3, 4, 5, 6} and each method (K-Means, Hierarchical, GMM), compute
Silhouette, Davies-Bouldin, and Calinski-Harabász. Present results as a tidy
DataFrame sorted by Silhouette descending.

**Based on your table**: Which (method, k) combination would you recommend to
a marketing team and why?

In [ ]:
# Exercise 6.1 — Full grid search: (method, k) x internal metrics
grid_rows = []
for k in [3, 4, 5, 6]:
    candidates = {
        'K-Means':             KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE).fit_predict(X),
        'Hierarchical (Ward)': AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X),
        'GMM (full)':          GaussianMixture(n_components=k, covariance_type='full', n_init=3,
                                               random_state=RANDOM_STATE).fit(X).predict(X),
    }
    for method, lab in candidates.items():
        shares = np.bincount(lab) / len(lab)
        grid_rows.append({'Method': method, 'k': k,
                          'Silhouette ↑': silhouette_score(X, lab),
                          'Davies-Bouldin ↓': davies_bouldin_score(X, lab),
                          'Calinski-Harabasz ↑': calinski_harabasz_score(X, lab),
                          'Smallest cluster %': 100 * shares.min()})

df_grid = (pd.DataFrame(grid_rows)
           .sort_values('Silhouette ↑', ascending=False)
           .reset_index(drop=True)
           .round(4))
display(df_grid)
best = df_grid.iloc[0]
print(f"Best by silhouette: {best['Method']} with k={best['k']}")
print("Rank by each metric (1 = best):")
ranks = pd.DataFrame({'Method': df_grid['Method'], 'k': df_grid['k'],
                      'rank Sil': df_grid['Silhouette ↑'].rank(ascending=False),
                      'rank DB': df_grid['Davies-Bouldin ↓'].rank(ascending=True),
                      'rank CH': df_grid['Calinski-Harabasz ↑'].rank(ascending=False)})
ranks['mean rank'] = ranks[['rank Sil', 'rank DB', 'rank CH']].mean(axis=1)
print(ranks.sort_values('mean rank').head(5).to_string(index=False))

**Recommendation.** The table is led by **K-Means with k = 3** (closely followed by Ward with k = 3): highest silhouette, lowest Davies-Bouldin and highest Calinski-Harabasz, and all metrics degrade monotonically as k grows, for every method. GMM ranks lower on all three metrics for the reasons discussed above (these metrics favour spherical, hard clusters). For a marketing team I would recommend **K-Means, k = 3**: (i) it is the best or near-best on all internal metrics simultaneously; (ii) the three segments map directly onto interpretable RFM personas (Champions / Regular / Lapsed) with no tiny, un-actionable cluster (see "Smallest cluster %"); (iii) K-Means has a `predict` method, so new customers can be assigned to a segment without refitting — something Ward cannot do. If the team wants a fourth segment (e.g. to separate "new customers" from "regulars"), k = 4 with K-Means is the next option, accepting a lower silhouette. GMM (k = 3) remains useful as a complement to flag uncertain/transition customers (Exercise 5.2).

## Section 7: Dataset 2 — HAR & External Evaluation (ARI / NMI) <a id='7'></a>

Internal metrics measure cluster *geometry*, but they cannot tell you whether the
clusters align with meaningful real-world categories.  When ground-truth labels
exist, use **external metrics**:

| Metric | Range | Meaning |
|---|---|---|
| **ARI** (Adjusted Rand Index) | −1 … 1 | 0 = random, 1 = perfect match, adjusted for chance |
| **NMI** (Normalised Mutual Info) | 0 … 1 | information shared between cluster and true labels |

### Human Activity Recognition (HAR)

Collected from 30 volunteers wearing a Samsung Galaxy S II on the waist.
Accelerometer and gyroscope measurements at 50 Hz → 561 engineered features.
**6 activity labels**: WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS,
SITTING, STANDING, LAYING.

In [ ]:
print("Loading HAR from OpenML (cached after first download)…")
har = fetch_openml('har', version=1, as_frame=False, parser='auto')
X_har_raw = har.data.astype(float)

# OpenML 'har' encodes activities as numeric codes '1'..'6'.
# Map them to the standard UCI HAR activity names.
HAR_ACTIVITY = {
    '1': 'WALKING', '2': 'WALKING_UPSTAIRS', '3': 'WALKING_DOWNSTAIRS',
    '4': 'SITTING', '5': 'STANDING', '6': 'LAYING',
}
y_har_str = np.array([HAR_ACTIVITY[c] for c in har.target])

# Map string labels → integers
activity_names = sorted(np.unique(y_har_str))
label_map = {name: i for i, name in enumerate(activity_names)}
y_har = np.array([label_map[l] for l in y_har_str])

print(f"Shape: {X_har_raw.shape}")
print(f"Activities: {activity_names}")
pd.Series(y_har_str).value_counts()

In [ ]:
# Scale -> PCA (retain 95% of the variance)
print("Scaling and reducing with PCA…")
scaler_har = StandardScaler()
X_har_scaled = scaler_har.fit_transform(X_har_raw)

pca_har = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_har_pca = pca_har.fit_transform(X_har_scaled)
print(f"Retained {pca_har.n_components_} PCs "
      f"out of {X_har_scaled.shape[1]} features "
      f"({pca_har.explained_variance_ratio_.sum():.1%} variance)")

In [ ]:
print("Running t-SNE (PCA dim → 2 dims) — may take ~2 minutes…")
tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE,
            n_jobs=-1, n_iter=1000)
X_har_tsne = tsne.fit_transform(X_har_pca)
print("Done.")

In [ ]:
# t-SNE coloured by true activity
colors = plt.cm.tab10(np.linspace(0, 0.6, len(activity_names)))

fig, ax = plt.subplots(figsize=(10, 7))
for i, (name, color) in enumerate(zip(activity_names, colors)):
    mask = y_har == i
    ax.scatter(X_har_tsne[mask, 0], X_har_tsne[mask, 1],
               c=[color], alpha=0.4, s=12, label=name)

ax.set_title('HAR — t-SNE projection coloured by true activity label', fontsize=13)
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9, markerscale=2)
plt.tight_layout()
plt.show()

In [ ]:
# Fit all three clustering methods (k = number of activities = 6)
N_ACT = len(activity_names)

km_har  = KMeans(n_clusters=N_ACT, init='k-means++', n_init=10, random_state=RANDOM_STATE)
agg_har = AgglomerativeClustering(n_clusters=N_ACT, linkage='ward')
gmm_har = GaussianMixture(n_components=N_ACT, covariance_type='full',
                           random_state=RANDOM_STATE, n_init=3)

pred_km  = km_har.fit_predict(X_har_pca)
pred_agg = agg_har.fit_predict(X_har_pca)
pred_gmm = gmm_har.fit_predict(X_har_pca)

har_rows = []
for name, pred in [('K-Means', pred_km),
                    ('Hierarchical (Ward)', pred_agg),
                    ('GMM', pred_gmm)]:
    har_rows.append({
        'Method':              name,
        'ARI ↑':               adjusted_rand_score(y_har, pred),
        'NMI ↑':               normalized_mutual_info_score(y_har, pred),
        'Silhouette ↑':        silhouette_score(X_har_pca, pred),
        'Davies-Bouldin ↓':    davies_bouldin_score(X_har_pca, pred),
    })

pd.DataFrame(har_rows).set_index('Method').round(4)

**Discussion**: Do the internal metrics (Silhouette, Davies-Bouldin) agree with
the external metrics (ARI, NMI) about which method is best?  This is the central
lesson of this section: internal metrics measure geometry, external metrics measure
*semantic alignment*.

### Exercise 7.1 — ARI / NMI vs k (Intermediate)

Using K-Means on the PCA-reduced HAR features, sweep k from 2 to 10.
For each k, compute ARI, NMI, and Silhouette.  Plot all three as a function of k
on a 1 × 3 subplot grid.

**Key question**: Does the Silhouette score peak at k = 6 (the true number of
activities)?  Does ARI?  Why might they disagree?

In [ ]:
# Exercise 7.1 — ARI / NMI / Silhouette vs k on HAR (K-Means on PCA features)
k_range_har = list(range(2, 11))
har_k_rows = []
for k in k_range_har:
    lab = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE).fit_predict(X_har_pca)
    har_k_rows.append({'k': k,
                       'ARI': adjusted_rand_score(y_har, lab),
                       'NMI': normalized_mutual_info_score(y_har, lab),
                       'Silhouette': silhouette_score(X_har_pca, lab)})
df_har_k = pd.DataFrame(har_k_rows).set_index('k')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, color in zip(axes, ['ARI', 'NMI', 'Silhouette'], ['tab:blue', 'tab:green', 'tab:orange']):
    ax.plot(df_har_k.index, df_har_k[metric], 'o-', color=color, linewidth=2)
    ax.axvline(N_ACT, color='gray', ls=':', label=f'true number of activities = {N_ACT}')
    best_k_m = df_har_k[metric].idxmax()
    ax.scatter([best_k_m], [df_har_k[metric].max()], s=150, facecolors='none', edgecolors='red', linewidths=2,
               label=f'peak at k = {best_k_m}')
    ax.set_xlabel('k'); ax.set_ylabel(metric); ax.set_title(f'{metric} vs k'); ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('HAR — K-Means on PCA features', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(df_har_k.round(4).to_string())
for metric in ['ARI', 'NMI', 'Silhouette']:
    print(f"{metric:>10s} peaks at k = {df_har_k[metric].idxmax()}")

**Does the silhouette peak at k = 6? Does ARI?** No — the silhouette peaks at **k = 2** (see the printed peaks): the strongest geometric structure in the HAR feature space is the split between **dynamic** activities (walking, upstairs, downstairs) and **static** ones (sitting, standing, laying), which are very far apart in sensor space. Splitting further only cuts these two large blobs into touching sub-clusters, so cohesion/separation (silhouette) gets worse. ARI and NMI peak at a larger k, closer to the true number of activities (but not necessarily exactly at 6), because they reward agreement with the 6 labels and penalise merging activities together.

**Why do they disagree?** Silhouette is an *internal* metric that only measures geometry (compact, well-separated, convex clusters), whereas ARI/NMI are *external* metrics that measure agreement with human-defined categories. The activities are not equally separable: several pairs (sitting vs standing, walking vs walking upstairs/downstairs) overlap heavily in feature space, so the "natural" geometric clusters do not coincide one-to-one with the semantic labels. The best k for geometry is therefore not the best k for semantics — which is the central lesson of this section: without labels, silhouette would have led us to the wrong number of activities.

### Exercise 7.2 — Static Activities Only (Advanced)

The six HAR activities split naturally into *dynamic* (WALKING, WALKING_UPSTAIRS,
WALKING_DOWNSTAIRS) and *static* (SITTING, STANDING, LAYING).

1. Filter the dataset to the three static activities only.
2. Fit K-Means with k = 3.
3. Compute ARI and NMI; compare to the full-dataset result.

**Hypothesis**: Static activities should be harder to distinguish (similar
body postures, low-frequency signals).  Does the data support this?

In [ ]:
# Exercise 7.2 — Static activities only (SITTING, STANDING, LAYING)
static_acts = ['SITTING', 'STANDING', 'LAYING']
dynamic_acts = ['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS']
mask_static = np.isin(y_har_str, static_acts)
mask_dynamic = np.isin(y_har_str, dynamic_acts)

def kmeans_subset(mask, k=3):
    lab = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE).fit_predict(X_har_pca[mask])
    return lab, adjusted_rand_score(y_har[mask], lab), normalized_mutual_info_score(y_har[mask], lab)

lab_static, ari_static, nmi_static = kmeans_subset(mask_static)
lab_dynamic, ari_dynamic, nmi_dynamic = kmeans_subset(mask_dynamic)
ari_full = adjusted_rand_score(y_har, pred_km)          # K-Means k=6 on all activities (Section 7)
nmi_full = normalized_mutual_info_score(y_har, pred_km)

cmp_static = pd.DataFrame({
    'All 6 activities (k=6)':     {'n samples': len(y_har), 'ARI': ari_full, 'NMI': nmi_full},
    'Static only (k=3)':          {'n samples': int(mask_static.sum()), 'ARI': ari_static, 'NMI': nmi_static},
    'Dynamic only (k=3)':         {'n samples': int(mask_dynamic.sum()), 'ARI': ari_dynamic, 'NMI': nmi_dynamic},
}).T
print(cmp_static.round(4).to_string())

print("\nContingency table — static activities vs K-Means clusters:")
print(pd.crosstab(pd.Series(y_har_str[mask_static], name='True activity'),
                  pd.Series(lab_static, name='Cluster')).to_string())

# Visual check on the t-SNE embedding computed in Section 7
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for name in static_acts:
    m = y_har_str[mask_static] == name
    axes[0].scatter(X_har_tsne[mask_static][m, 0], X_har_tsne[mask_static][m, 1], s=8, alpha=0.5, label=name)
axes[0].set_title('Static activities — true labels (t-SNE)'); axes[0].legend(markerscale=3)
axes[1].scatter(X_har_tsne[mask_static][:, 0], X_har_tsne[mask_static][:, 1], c=lab_static, cmap='tab10', s=8, alpha=0.5)
axes[1].set_title(f'Static activities — K-Means k=3 (ARI = {ari_static:.3f})')
plt.tight_layout(); plt.show()

**Is the hypothesis supported?** Partly — and the contingency table shows exactly where it holds. **LAYING** is almost perfectly isolated in its own cluster (gravity acts on a different axis of the phone when the subject is lying down, so its features are very different). **SITTING and STANDING**, however, are heavily mixed across the remaining clusters: both are upright, motionless postures that produce almost identical low-frequency accelerometer/gyroscope signals. As a result, the ARI/NMI on the static subset is driven by LAYING; the SITTING/STANDING confusion confirms the hypothesis that some static activities are **very hard to separate** without supervision. Comparing with the dynamic-only subset (same k = 3) and the full 6-class result in the table gives the overall picture: the difficulty is concentrated in specific pairs of postures (sitting vs standing, and to a lesser degree the walking variants), not in the static group as a whole.

## Section 8: Real-World Considerations <a id='8'></a>

### Preprocessing is decisive

Feature scaling is non-negotiable for distance-based methods (K-Means,
Hierarchical).  Skewed distributions (like Monetary) should be log-transformed
before scaling.  PCA before clustering reduces noise and speeds up computation —
particularly important for high-dimensional data like HAR.

### Choosing an algorithm

| Algorithm | Strengths | Weaknesses |
|---|---|---|
| K-Means | Fast, scalable, easy to interpret | Assumes spherical clusters, sensitive to init |
| Hierarchical | No k required up front, produces dendrogram, deterministic | O(n²) memory/time, no predict() for new data |
| GMM | Soft assignments, flexible covariance shapes | Slower, can collapse/overfit, needs good init |

### Internal vs external validation

Use **internal** metrics (Silhouette, Davies-Bouldin) when you have no labels —
which is the typical case.  If you can label even a small validation set, compute
**external** metrics (ARI, NMI) on it — they are strictly more informative.

### Scalability

For very large datasets:
- Replace `KMeans` with `MiniBatchKMeans` (sklearn, same API)
- Hierarchical clustering is impractical above ~50 k samples; use BIRCH or HDBSCAN
- GMM with `diag` covariance scales better than `full`

### Communicating results

Cluster labels are arbitrary integers.  Always *profile* each cluster (mean RFM
values, mode activity) and give it a human-readable name before presenting to
stakeholders.

## Section 9: Final Challenges (optional) <a id='9'></a>

### Challenge 9.1 — Image Colour Quantization

K-Means can compress an image by replacing each pixel colour with its nearest
cluster centroid — effectively reducing the palette to k colours.

Apply K-Means to the pixels of sklearn's built-in `china.jpg` sample image with
k ∈ {2, 4, 8, 16}. Display the original and all four quantized versions.

In [ ]:
# Challenge 9.1 — Image colour quantization with K-Means
china = load_sample_image('china.jpg')
img = np.array(china, dtype=np.float64) / 255.0           # (h, w, 3) in [0, 1]
h, w, c = img.shape
pixels = img.reshape(-1, c)

# Fit on a random subsample of pixels (fast), then assign every pixel to its nearest centroid
pixel_sample = shuffle(pixels, random_state=RANDOM_STATE, n_samples=10_000)
quantized = {}
for k in [2, 4, 8, 16]:
    km_img = KMeans(n_clusters=k, n_init=4, random_state=RANDOM_STATE).fit(pixel_sample)
    quantized[k] = km_img.cluster_centers_[km_img.predict(pixels)].reshape(h, w, c)

fig, axes = plt.subplots(1, 5, figsize=(24, 5))
axes[0].imshow(img); axes[0].set_title(f'Original ({len(np.unique(pixels, axis=0)):,} colours)')
for ax, k in zip(axes[1:], [2, 4, 8, 16]):
    ax.imshow(quantized[k]); ax.set_title(f'K-Means, k = {k} colours')
for ax in axes:
    ax.axis('off')
plt.tight_layout(); plt.show()

**Observation.** With k = 2 only the light/dark structure survives; with k = 4–8 the main regions (sky, building, vegetation) become recognisable, and with k = 16 the image is visually close to the original even though it uses 16 colours instead of tens of thousands — each pixel can be stored as a 4-bit index into a 16-colour palette instead of 24 bits, a ~6× compression of the colour information.

### Challenge 9.2 — Beat the Benchmark (HAR)

The best ARI you achieved in Section 7 is your benchmark.

Try to improve it by experimenting with:
- Different numbers of PCA components (10, 100, 200)
- Different clustering methods or hyperparameters
- Applying UMAP instead of PCA for dimensionality reduction (requires `umap-learn`)

Report your best (method, preprocessing, ARI, NMI) and describe what worked.

In [ ]:
# Challenge 9.2 — Beat the benchmark (HAR)
!pip install umap-learn --quiet
import time
import umap

benchmark = pd.DataFrame(har_rows).set_index('Method')
bench_ari = benchmark['ARI ↑'].max()
print(f"Benchmark (Section 7): best ARI = {bench_ari:.4f} ({benchmark['ARI ↑'].idxmax()}, PCA 95% = {pca_har.n_components_} PCs)")

def evaluate(rep, name):
    rows = []
    for method, model in [('K-Means', KMeans(n_clusters=N_ACT, n_init=10, random_state=RANDOM_STATE)),
                          ('GMM full', GaussianMixture(n_components=N_ACT, covariance_type='full', n_init=2,
                                                       random_state=RANDOM_STATE, reg_covar=1e-4))]:
        t0 = time.time()
        lab = model.fit_predict(rep)
        rows.append({'Preprocessing': name, 'Method': method,
                     'ARI': adjusted_rand_score(y_har, lab), 'NMI': normalized_mutual_info_score(y_har, lab),
                     'time (s)': time.time() - t0})
    return rows

experiments = []
for n_comp in [10, 100, 200]:
    rep = PCA(n_components=n_comp, random_state=RANDOM_STATE).fit_transform(X_har_scaled)
    experiments += evaluate(rep, f'PCA-{n_comp}')

# Non-linear manifold reduction: UMAP (on the standardised features)
for n_neighbors in [15, 50]:
    rep_umap = umap.UMAP(n_components=10, n_neighbors=n_neighbors, min_dist=0.0,
                         random_state=RANDOM_STATE).fit_transform(X_har_scaled)
    experiments += evaluate(rep_umap, f'UMAP-10 (n_neighbors={n_neighbors})')

df_exp = pd.DataFrame(experiments).sort_values('ARI', ascending=False).reset_index(drop=True).round(4)
display(df_exp)
best_exp = df_exp.iloc[0]
print(f"Best: {best_exp['Method']} + {best_exp['Preprocessing']} -> ARI = {best_exp['ARI']:.4f}, NMI = {best_exp['NMI']:.4f} "
      f"({'beats' if best_exp['ARI'] > bench_ari else 'does not beat'} the benchmark {bench_ari:.4f})")

**Results.** The table ranks every (preprocessing, method) combination by ARI and the last line reports the best one against the Section 7 benchmark. The PCA experiments confirm that the number of components matters only up to a point: PCA-10 keeps the coarse dynamic/static structure but loses detail, while PCA-100 and PCA-200 behave similarly to the 95 %-variance benchmark (extra components mostly add noise and runtime). The largest change comes from the **representation**: UMAP builds a non-linear embedding that preserves local neighbourhoods, so samples of the same activity are pulled into compact groups and K-Means/GMM — which assume roughly convex clusters — work much better on it. The remaining errors are again concentrated in SITTING vs STANDING, which overlap even in the UMAP space. Conclusion: for this dataset, changing the representation (UMAP) has a much bigger effect on ARI/NMI than changing the clustering algorithm.

**What worked.** A few robust patterns usually emerge:

- **Too few PCA components throws away signal.** PCA-10 captures less variance and
  typically loses ARI versus the 95%-variance benchmark from Section 7.
- **More components help only up to a point.** Once enough variance is retained,
  adding further PCs mostly piles on noise dimensions; ARI plateaus or dips
  slightly while runtime grows. The 561 HAR features are highly correlated, so the
  useful structure lives in a modest number of directions.
- **GMM vs K-Means.** With `covariance_type='full'`, GMM can model elongated,
  correlated clusters that K-Means (which assumes spherical clusters) cannot —
  it sometimes edges ahead, but is slower and more sensitive to initialisation.
- **UMAP is usually the biggest win.** A non-linear manifold reduction pulls the
  six activities into more compact, better-separated groups than linear PCA, so
  K-Means on a 10-D UMAP embedding often gives the highest ARI of all. The lesson:
  *the preprocessing/representation matters more than the clustering algorithm.*

### Challenge 9.3 — Country Segment Comparison (Online Retail)

The Online Retail dataset includes a `Country` column.

1. Repeat the full RFM pipeline separately for the **United Kingdom** (the
   dominant country, ~90% of records) and one other country with enough
   customers (e.g., Germany or France).
2. Fit K-Means with k = K_BEST on each subset.
3. Compare the cluster profiles (mean RFM per cluster).

**Do purchasing patterns differ significantly across countries?**

In [ ]:
# Challenge 9.3 — Country segment comparison (United Kingdom vs Germany/France)
def rfm_for_country(df_tx, country, k=K_BEST):
    d = df_tx[df_tx['Country'] == country]
    r = d.groupby('CustomerID').agg(
        Recency=('InvoiceDate', lambda x: (reference_date - x.max()).days),   # same reference date for both
        Frequency=('InvoiceNo', 'nunique'),
        Monetary=('TotalPrice', 'sum'),
    ).reset_index()
    r_log = r[['Recency', 'Frequency', 'Monetary']].copy()
    r_log[log_features] = np.log1p(r_log[log_features])
    Xc = StandardScaler().fit_transform(r_log)
    km_c = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    r['Cluster'] = km_c.fit_predict(Xc)
    # Relabel clusters by mean Monetary so that segment 0 = lowest value ... k-1 = highest value
    order = r.groupby('Cluster')['Monetary'].mean().sort_values().index
    r['Segment'] = r['Cluster'].map({c: i for i, c in enumerate(order)})
    prof = r.groupby('Segment').agg(Customers=('CustomerID', 'count'), Recency=('Recency', 'mean'),
                                    Frequency=('Frequency', 'mean'), Monetary=('Monetary', 'mean'))
    prof['% customers'] = 100 * prof['Customers'] / len(r)
    return r, prof, silhouette_score(Xc, r['Cluster'])

customers_per_country = df.groupby('Country')['CustomerID'].nunique().sort_values(ascending=False)
print("Customers per country (top 5):"); print(customers_per_country.head().to_string())
other_country = 'Germany' if customers_per_country.get('Germany', 0) >= customers_per_country.get('France', 0) else 'France'

profiles, rfm_country = {}, {}
for country in ['United Kingdom', other_country]:
    rfm_country[country], profiles[country], sil_c = rfm_for_country(df, country)
    print(f"\n{country}: {len(rfm_country[country])} customers | silhouette (k={K_BEST}) = {sil_c:.3f}")
    print(profiles[country].round(1).to_string())

seg_names = {0: 'Low value', 1: 'Mid value', 2: 'High value'} if K_BEST == 3 else {i: f'Segment {i}' for i in range(K_BEST)}
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
width = 0.38
for j, feat in enumerate(['Recency', 'Frequency', 'Monetary', '% customers']):
    for i, country in enumerate(profiles):
        p = profiles[country]
        axes[j].bar(np.arange(K_BEST) + (i - 0.5) * width, p[feat], width, label=country)
    axes[j].set_xticks(range(K_BEST)); axes[j].set_xticklabels([seg_names[s] for s in range(K_BEST)])
    axes[j].set_title(f'Mean {feat} per segment' if feat != '% customers' else 'Segment size (%)')
    if feat == 'Monetary':
        axes[j].set_yscale('log')
axes[0].legend()
plt.suptitle(f'RFM segment profiles — United Kingdom vs {other_country}', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("\nOverall medians per country:")
print(pd.DataFrame({c: rfm_country[c][['Recency', 'Frequency', 'Monetary']].median() for c in rfm_country}).round(1).to_string())

**Do purchasing patterns differ across countries?** The **structure** of the segments is the same in both countries: with k = 3 we obtain a low-value/lapsed segment (high Recency, ~1–2 orders, low spend), a regular segment and a high-value segment (recent, frequent, high spend), so the RFM segmentation transfers well across markets. The **levels and proportions** differ, though: the foreign market has far fewer customers (≈ 90–95 vs ≈ 3 900 in the UK), and its customers tend to be wholesale-like buyers with higher spend per customer and a larger share in the mid/high-value segments, while the UK has a long tail of small, occasional domestic buyers (larger low-value segment, more extreme top spenders). Two caveats: with fewer than 100 customers the foreign clustering is much less stable (a few customers can move a centroid), and the segment means are dominated by a few very large accounts. In practice, a single global model with a country indicator — or country-specific thresholds — would be preferable to treating the small market with its own clustering.

## Summary and Key Takeaways <a id='10'></a>

| Concept | Takeaway |
|---|---|
| RFM engineering | Raw transactions must be aggregated into per-entity features before clustering |
| Scaling | Always scale before K-Means or Hierarchical; GMM is less sensitive but still benefits |
| Elbow + Silhouette | Use both together — elbow finds the bend, silhouette confirms |
| Ward linkage | Default first choice for agglomerative clustering; produces balanced clusters |
| BIC/AIC | Prefer BIC for GMM; lower = better fit adjusted for complexity |
| Internal vs external | Internal metrics measure geometry; external metrics measure semantic alignment |
| ARI/NMI | Use when any ground-truth labels are available, even for a validation subset |

### Resources

- [scikit-learn clustering user guide](https://scikit-learn.org/stable/modules/clustering.html)
- [UCI Online Retail dataset](https://archive.ics.uci.edu/ml/datasets/Online+Retail)
- [UCI HAR dataset](https://archive.ics.uci.edu/ml/datasets/Human+Activity+Recognition+Using+Smartphones)
- Kaufman & Rousseeuw (1990) — *Finding Groups in Data* (silhouette score)
- McLachlan & Basford (1988) — *Mixture Models* (GMM theory)